# Does AI Sell? Hype Cues in Kickstarter Technology Campaigns

Input files: `muestra_final.xlsx`, `text_ai.xlsx`, `text_ai_llm.xlsx` and the five `estrato_*.csv` files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

## Data and variables

In [ ]:
df = pd.read_excel('muestra_final.xlsx')
print(df.shape)

In [ ]:
df['success'] = (df['estado'] == 'successful').astype(int)
df['pct_funded'] = df['recaudado'] / df['objetivo'] * 100
df['lanzamiento'] = pd.to_datetime(df['lanzamiento'])
df['fin'] = pd.to_datetime(df['fin'])
df['launch_year'] = df['lanzamiento'].dt.year
df['campaign_days'] = (df['fin'] - df['lanzamiento']).dt.days
df['us'] = (df['pais'] == 'US').astype(int)

df['rate'] = df['recaudado_usd'] / df['recaudado']
df['rate'] = df['rate'].fillna(df.groupby('moneda')['rate'].transform('median'))
eur_rate = df[df['moneda'] == 'EUR']['rate'].median()
df.loc[df['moneda'] == 'DKK', 'rate'] = eur_rate / 7.46
df['goal_usd'] = df['objetivo'] * df['rate']

df['log_goal'] = np.log(df['goal_usd'])
df['log_words'] = np.log1p(df['story_palabras'])
df['log_sharpness'] = np.log1p(df['nitidez'])
df['log_pct_funded'] = np.log1p(df['pct_funded'])

counts = df['subcategoria'].value_counts()
df['subcat'] = df['subcategoria'].where(df['subcategoria'].map(counts) >= 30, 'Other')
print(df['subcat'].value_counts())

## Exploratory analysis

In [ ]:
print('Duplicated projects:', df['id'].duplicated().sum())
print('Missing values:', df.drop(columns='error').isna().sum().sum())
print('Projects with scraping error:', df['error'].notna().sum())

In [ ]:
cols = ['objetivo', 'recaudado_usd', 'pct_funded', 'backers', 'campaign_days',
        'story_palabras', 'story_imagenes', 'story_videos', 'creador_proyectos_previos',
        'mention', 'mentions_ai', 'estetica', 'nitidez', 'brillo', 'contraste', 'colorido']
df[cols].describe().T

In [ ]:
df.groupby('success')[cols].mean().T

In [ ]:
df.groupby('mention')['success'].mean().plot(kind='bar')
plt.title('Success rate by AI mention (script)')
plt.show()

df.groupby('mentions_ai')['success'].mean().plot(kind='bar')
plt.title('Success rate by AI mention (LLM)')
plt.show()

In [ ]:
print(df.groupby('launch_year')[['mention', 'mentions_ai']].mean())

df.groupby('launch_year')[['mention', 'mentions_ai']].mean().plot()
plt.title('Share of projects mentioning AI by year')
plt.show()

In [ ]:
df.groupby('launch_year')['success'].mean()

In [ ]:
df[['estetica', 'nitidez', 'brillo', 'contraste', 'colorido']].hist(bins=30, figsize=(12, 6))
plt.show()

In [ ]:
df.boxplot(column='estetica', by='success')
plt.show()

df.boxplot(column='estetica', by='mention')
plt.show()

In [ ]:
corr_cols = ['success', 'pct_funded', 'objetivo', 'campaign_days', 'mention', 'mentions_ai',
             'story_palabras', 'story_imagenes', 'story_videos', 'creador_proyectos_previos',
             'estetica', 'nitidez', 'brillo', 'contraste', 'colorido']
df[corr_cols].corr(method='spearman').round(2)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(df[corr_cols].corr(method='spearman'), cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar()
plt.show()

In [ ]:
df.groupby('subcategoria').agg(projects=('id', 'count'),
                               success_rate=('success', 'mean'),
                               ai_share=('mention', 'mean')).sort_values('projects', ascending=False)

## Image processing check

In [ ]:
print('Projects without aesthetic score:', df['estetica'].isna().sum())
print('Projects without OpenCV metrics:', df['nitidez'].isna().sum())

## Population and sample by stratum

In [ ]:
files = {'<25%': 'estrato_menos25.csv', '25-50%': 'estrato_25-50.csv', '50-75%': 'estrato_50-75.csv',
         '75-100%': 'estrato_75-100.csv', '>100%': 'estrato_mas100.csv'}
parts = []
for name, f in files.items():
    d = pd.read_csv(f, sep=';', encoding='utf-8-sig')
    d['estrato'] = name
    parts.append(d)
pop = pd.concat(parts).drop_duplicates(subset='id')

cutoff = pop.groupby('estrato')['lanzamiento'].min().max()
pop = pop[pop['lanzamiento'] >= cutoff]
print('Window start:', cutoff)
print('Window end (latest launch):', pop['lanzamiento'].max())
print('Projects in the window (population):', len(pop))
print('Sample size:', len(df))

In [ ]:
table = pd.DataFrame({'Projects in the window': pop['estrato'].value_counts(),
                      'Projects in the sample': df['estrato'].value_counts()})
table = table.loc[['<25%', '25-50%', '50-75%', '75-100%', '>100%']]
table.loc['Total'] = table.sum()
table

## Language model fallback

In [ ]:
llm = pd.read_excel('text_ai_llm.xlsx')
llm = llm[llm['id'].isin(df['id'])]
print('Projects classified with the fallback (attempt 2):', (llm['attempt'] == 2).sum())

## Descriptive statistics

In [ ]:
print('Projects:', len(df))
print('First launch:', df['lanzamiento'].min().date())
print('Last launch:', df['lanzamiento'].max().date())
print('Funded:', df['success'].sum())
print('Success rate:', round(df['success'].mean() * 100, 1), '% (Kickstarter official: 24.9%)')
print(df['estado'].value_counts())

### Table 1

In [ ]:
cols = ['goal_usd', 'campaign_days', 'story_palabras', 'story_imagenes', 'story_videos',
        'creador_proyectos_previos', 'us', 'mention', 'mentions_ai',
        'estetica', 'nitidez', 'brillo', 'contraste', 'colorido']
binary = ['us', 'mention', 'mentions_ai']

rows = []
for c in cols:
    funded = df[df['success'] == 1][c]
    not_funded = df[df['success'] == 0][c]
    if c in binary:
        p = chi2_contingency(pd.crosstab(df[c], df['success']))[1]
    else:
        p = mannwhitneyu(funded, not_funded)[1]
    rows.append([c, df[c].count(), df[c].mean(), df[c].std(), df[c].median(), df[c].min(), df[c].max(),
                 not_funded.mean(), funded.mean(), p])

table1 = pd.DataFrame(rows, columns=['variable', 'N', 'mean', 'SD', 'median', 'min', 'max',
                                     'mean if not funded', 'mean if funded', 'p-value'])
table1.round(3)

In [ ]:
funded = df[df['success'] == 1]
not_funded = df[df['success'] == 0]

print('Median goal (USD): funded', round(funded['goal_usd'].median()), 'vs not funded', round(not_funded['goal_usd'].median()))
print('Median campaign days: funded', funded['campaign_days'].median(), 'vs not funded', not_funded['campaign_days'].median())
print('% with previous projects: funded', round((funded['creador_proyectos_previos'] > 0).mean() * 100, 1),
      'vs not funded', round((not_funded['creador_proyectos_previos'] > 0).mean() * 100, 1))
print('Median aesthetic: funded', round(funded['estetica'].median(), 2), 'vs not funded', round(not_funded['estetica'].median(), 2))
print('Mann-Whitney p (aesthetic):', mannwhitneyu(funded['estetica'], not_funded['estetica'])[1])

## AI mentions

In [ ]:
text = pd.read_excel('text_ai.xlsx')
df = df.merge(text[['id', 'ai_strict_count']], on='id', how='left')

print('% mention AI (keywords):', round(df['mention'].mean() * 100, 1))
print('% explicit term (strict tier):', round((df['ai_strict_count'] > 0).mean() * 100, 1))

In [ ]:
by_year = df.groupby('launch_year')['mention'].agg(['count', 'mean'])
by_year['mean'] = by_year['mean'] * 100
print(by_year)

(df.groupby('launch_year')['mention'].mean() * 100).plot(kind='bar')
plt.ylabel('% of projects that mention AI')
plt.xlabel('Launch year')
plt.show()

In [ ]:
print('% mention AI (LLM):', round(df['mentions_ai'].mean() * 100, 1))
print('Agreement:', round((df['mention'] == df['mentions_ai']).mean() * 100, 1), '%')
print('Cohen kappa:', round(cohen_kappa_score(df['mention'], df['mentions_ai']), 3))
print(pd.crosstab(df['mention'], df['mentions_ai'], rownames=['keywords'], colnames=['LLM']))

In [ ]:
print('Success rate if mention = 1:', round(df[df['mention'] == 1]['success'].mean() * 100, 1))
print('Success rate if mention = 0:', round(df[df['mention'] == 0]['success'].mean() * 100, 1))
print('Chi-square p:', chi2_contingency(pd.crosstab(df['mention'], df['success']))[1])

(df.groupby('mention')['success'].mean() * 100).plot(kind='bar')
plt.ylabel('% funded')
plt.xlabel('Mentions AI (0 = no, 1 = yes)')
plt.show()

## Cover aesthetics

In [ ]:
print('Min:', round(df['estetica'].min(), 2))
print('Max:', round(df['estetica'].max(), 2))
print('Mean:', round(df['estetica'].mean(), 2))

In [ ]:
m_aes = smf.logit('success ~ estetica', data=df).fit(cov_type='HC1')
print('Coefficient:', round(m_aes.params['estetica'], 3))
print('p-value:', m_aes.pvalues['estetica'])
print('Odds ratio:', round(np.exp(m_aes.params['estetica']), 2))
print('Marginal effect (pp):', round(m_aes.get_margeff().margeff[0] * 100, 1))

In [ ]:
m_aes2 = smf.logit('success ~ estetica + log_sharpness + brillo + contraste + colorido', data=df).fit(cov_type='HC1')
print(m_aes2.summary())

## Logistic regression models

In [ ]:
m1_vars = 'mention + estetica'
m2_vars = m1_vars + ' + log_sharpness + brillo + contraste + colorido'
m3_vars = m2_vars + ' + log_goal + campaign_days + log_words + story_imagenes + story_videos + creador_proyectos_previos + us'
m4_vars = m3_vars + ' + C(subcat)'

m1 = smf.logit('success ~ ' + m1_vars, data=df).fit(cov_type='HC1')
m2 = smf.logit('success ~ ' + m2_vars, data=df).fit(cov_type='HC1')
m3 = smf.logit('success ~ ' + m3_vars, data=df).fit(cov_type='HC1')
m4 = smf.logit('success ~ ' + m4_vars, data=df).fit(cov_type='HC1', maxiter=100)

In [ ]:
m5_vars = m4_vars.replace(' + story_imagenes', '')
m5 = smf.logit('success ~ ' + m5_vars, data=df).fit(cov_type='HC1', maxiter=100)

In [ ]:
cols = []
for name, m in [('Model 1', m1), ('Model 2', m2), ('Model 3', m3), ('Model 4', m4), ('Model 5', m5)]:
    me = m.get_margeff().summary_frame()
    col = me['dy/dx'].round(3).astype(str) + ' (' + me['Std. Err.'].round(3).astype(str) + ')'
    col = col + me['Pr(>|z|)'].apply(lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else '')
    col['N'] = str(int(m.nobs))
    col['Pseudo R2'] = str(round(m.prsquared, 3))
    col['AUC'] = str(round(roc_auc_score(df['success'], m.predict(df)), 3))
    col.name = name
    cols.append(col)

table2 = pd.concat(cols, axis=1)
table2 = table2[~table2.index.str.startswith('C(subcat)')]
table2 = table2.fillna('')
stats = ['N', 'Pseudo R2', 'AUC']
table2 = table2.loc[[i for i in table2.index if i not in stats] + stats]
table2.to_excel('table2.xlsx')
table2

In [ ]:
pvalues = pd.concat([m1.get_margeff().summary_frame()['Pr(>|z|)'],
                     m2.get_margeff().summary_frame()['Pr(>|z|)'],
                     m3.get_margeff().summary_frame()['Pr(>|z|)'],
                     m4.get_margeff().summary_frame()['Pr(>|z|)'],
                     m5.get_margeff().summary_frame()['Pr(>|z|)']], axis=1)
pvalues.columns = ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5']
pvalues.round(3)

### Full output of Models 4 and 5

In [ ]:
print(m4.summary())
print(m5.summary())

In [ ]:
for v in ['story_imagenes', 'log_goal', 'us']:
    print(v, 'SD:', round(df[v].std(), 2), 'effect per SD:', round(m4.params[v] * df[v].std(), 2))

print(spearmanr(df['estetica'], df['story_imagenes']))

In [ ]:
me4 = m4.get_margeff().summary_frame()
me3 = m3.get_margeff().summary_frame()
print('Model 4, AI mention: ME', round(me4.loc['mention', 'dy/dx'] * 100, 2), 'pp, p =', round(me4.loc['mention', 'Pr(>|z|)'], 3))
print('Model 3, aesthetic: ME', round(me3.loc['estetica', 'dy/dx'] * 100, 2), 'pp, p =', round(me3.loc['estetica', 'Pr(>|z|)'], 3))
print('Model 4, aesthetic: ME', round(me4.loc['estetica', 'dy/dx'] * 100, 2), 'pp, p =', round(me4.loc['estetica', 'Pr(>|z|)'], 3))

In [ ]:
print('10% higher goal (pp):', round(me4.loc['log_goal', 'dy/dx'] * np.log(1.1) * 100, 2), ', p =', round(me4.loc['log_goal', 'Pr(>|z|)'], 3))
print('One more campaign day (pp):', round(me4.loc['campaign_days', 'dy/dx'] * 100, 2), ', p =', round(me4.loc['campaign_days', 'Pr(>|z|)'], 3))
print('One more previous project (pp):', round(me4.loc['creador_proyectos_previos', 'dy/dx'] * 100, 2), ', p =', round(me4.loc['creador_proyectos_previos', 'Pr(>|z|)'], 3))
print('US-based (pp):', round(me4.loc['us', 'dy/dx'] * 100, 2), ', p =', round(me4.loc['us', 'Pr(>|z|)'], 3))

## Random forest

In [ ]:
features = ['mention', 'estetica', 'log_sharpness', 'brillo', 'contraste', 'colorido',
            'log_goal', 'campaign_days', 'log_words', 'story_imagenes', 'story_videos',
            'creador_proyectos_previos', 'us', 'subcat']

X = df[features].copy()
X['subcat'] = X['subcat'].astype('category').cat.codes
y = df['success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', len(X_train), 'Test:', len(X_test))

In [ ]:
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)
print('Random forest AUC (test):', round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

train = df.loc[X_train.index]
test = df.loc[X_test.index]
logit_train = smf.logit('success ~ ' + m4_vars, data=train).fit(maxiter=100)
print('Logistic regression (Model 4) AUC (test):', round(roc_auc_score(test['success'], logit_train.predict(test)), 3))

In [ ]:
imp = permutation_importance(rf, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=42)
importance = pd.Series(imp.importances_mean, index=features).sort_values(ascending=False)
importance = importance.to_frame('importance')
importance['rank'] = range(1, len(importance) + 1)
print(importance)

importance.rename(index={'log_goal': 'Funding goal (log)', 'subcat': 'Subcategory', 'story_imagenes': 'Story images', 'story_videos': 'Story videos', 'creador_proyectos_previos': 'Previous projects', 'log_words': 'Story words (log)', 'log_sharpness': 'Cover sharpness (log)', 'campaign_days': 'Campaign days', 'contraste': 'Contrast', 'brillo': 'Brightness', 'estetica': 'Aesthetic score', 'colorido': 'Colourfulness', 'us': 'US creator', 'mention': 'AI mention'})['importance'].sort_values().plot(kind='barh')
plt.xlabel('Drop in AUC when the variable is shuffled')
plt.show()

### Random forest without story images

In [ ]:
features5 = [f for f in features if f != 'story_imagenes']

rf5 = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, random_state=42)
rf5.fit(X_train[features5], y_train)
print('Random forest without story images, AUC (test):', round(roc_auc_score(y_test, rf5.predict_proba(X_test[features5])[:, 1]), 3))

logit5_train = smf.logit('success ~ ' + m5_vars, data=train).fit(maxiter=100)
print('Logistic regression (Model 5) AUC (test):', round(roc_auc_score(test['success'], logit5_train.predict(test)), 3))

In [ ]:
labels = {'log_goal': 'Funding goal (log)', 'subcat': 'Subcategory', 'story_imagenes': 'Story images',
    'story_videos': 'Story videos', 'creador_proyectos_previos': 'Previous projects',
    'log_words': 'Story words (log)', 'log_sharpness': 'Cover sharpness (log)',
    'campaign_days': 'Campaign days', 'contraste': 'Contrast', 'brillo': 'Brightness',
    'estetica': 'Aesthetic score', 'colorido': 'Colourfulness', 'us': 'US creator', 'mention': 'AI mention'}

imp5 = permutation_importance(rf5, X_test[features5], y_test, scoring='roc_auc', n_repeats=10, random_state=42)
importance5 = pd.Series(imp5.importances_mean, index=features5).sort_values(ascending=False)
importance5 = importance5.to_frame('importance')
importance5['rank'] = range(1, len(importance5) + 1)
print(importance5)

importance5 = importance5.rename(index=labels)
importance5['importance'].sort_values().plot(kind='barh')
plt.xlabel('Drop in AUC when the variable is shuffled')
plt.show()

## Robustness checks

### LLM measure of AI

In [ ]:
m4_llm = smf.logit('success ~ ' + m4_vars.replace('mention', 'mentions_ai'), data=df).fit(cov_type='HC1', maxiter=100)
me = m4_llm.get_margeff().summary_frame()
print('LLM AI measure: ME', round(me.loc['mentions_ai', 'dy/dx'] * 100, 2), 'pp, p =', round(me.loc['mentions_ai', 'Pr(>|z|)'], 3))

### Without story images

In [ ]:
m4_noimg = smf.logit('success ~ ' + m4_vars.replace(' + story_imagenes', ''), data=df).fit(cov_type='HC1', maxiter=100)
print('Aesthetic coefficient with story images:', round(m4.params['estetica'], 3), ', p =', round(m4.pvalues['estetica'], 3))
print('Aesthetic coefficient without story images:', round(m4_noimg.params['estetica'], 3), ', p =', round(m4_noimg.pvalues['estetica'], 3))
print('AI coefficient without story images:', round(m4_noimg.params['mention'], 3), ', p =', round(m4_noimg.pvalues['mention'], 3))

### Without canceled projects

In [ ]:
no_cancel = df[df['estado'] != 'canceled']
print('Canceled projects excluded:', len(df) - len(no_cancel))

m4_nc = smf.logit('success ~ ' + m4_vars, data=no_cancel).fit(cov_type='HC1', maxiter=100)
print('AI mention: coef', round(m4_nc.params['mention'], 3), ', p =', round(m4_nc.pvalues['mention'], 3))
print('Aesthetic: coef', round(m4_nc.params['estetica'], 3), ', p =', round(m4_nc.pvalues['estetica'], 3))

### Percentage funded as the outcome

In [ ]:
ols = smf.ols('log_pct_funded ~ ' + m4_vars, data=df).fit(cov_type='HC1')
print('AI mention: coef', round(ols.params['mention'], 3), ', p =', round(ols.pvalues['mention'], 3))
print('Aesthetic: coef', round(ols.params['estetica'], 3), ', p =', round(ols.pvalues['estetica'], 3))
print('R2:', round(ols.rsquared, 3))